# Lab 4: LLMs and Prompt Engineering for Decision Support

**Duration:** 2 weeks [30 Jul - 13 Aug, 2026]
**Due Date:** 13th August, 2026
**Format:** Jupyter Notebook / Google Colab + external APIs + GitHub version control
**Grading:** This is a graded lab.

**Student Name:** Adelle Solange Naa Ayele Hammond
**Student ID:** 72942028

---

### Objective

In the previous labs you *trained* models. In this lab you will *use* a model that someone
else spent millions of dollars training — a **Large Language Model (LLM)** — and learn that
getting good results out of one is an engineering discipline of its own: **prompt
engineering**.

You will build a **decision support system for a microfinance loan officer**. Given a pile of
free-text loan application letters, your system will:

1. **Summarize** each application into a short, factual brief,
2. **Extract** specific structured data points (JSON) that a downstream system could store,
3. Produce a **decision-support recommendation** — while keeping the human firmly in the loop.

Just as importantly, you will **evaluate** the LLM's output for quality, reliability, and
appropriateness: Does it hallucinate? Is it consistent across runs? Should it be trusted to
make the final call?

---

### Choosing an API provider

You need an LLM API with a **free tier**. Recommended options (pick ONE):

| Provider | Free tier | Notes |
|---|---|---|
| **Groq** (recommended) | Yes, generous | OpenAI-compatible API, very fast, open models (Llama) |
| **Google Gemini** | Yes | `google-generativeai` package |
| **Hugging Face Inference API** | Yes, limited | Many open models |
| OpenAI / Anthropic | Paid | Fine if you already have credits |

The notebook's example code uses the **OpenAI-compatible chat format** (works with Groq and
OpenAI directly; Gemini users adapt the call in one place). Everything else in the lab is
provider-agnostic.

---
### Part 0: Repository and API-key setup

1. Create a **public** repository named `lab-4-llm-decision-support` and save this notebook
   inside it.
2. Sign up with your chosen provider and create an **API key**.
3. **NEVER hard-code or commit your API key.** This is a graded requirement.
   - Locally: put it in a `.env` file and add `.env` to `.gitignore`.
   - Colab: use the Secrets panel (key icon) and read it with `google.colab.userdata`.
4. Add a `requirements.txt`: `openai python-dotenv pandas matplotlib`.
5. Commit and push after **each Part** — we will check for incremental commits.

> **A leaked key in your commit history = resubmission + penalty.** Keys can be scraped from
> public repos within minutes.

In [20]:
%pip install python-dotenv
%pip install openai

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
# API-key setup — DO NOT hard-code your key in this cell.

import os

# --- Local (with a .env file) ---
from dotenv import load_dotenv
load_dotenv()
API_KEY = os.environ["GROQ_API_KEY"]

# --- Google Colab (Secrets panel) ---
# from google.colab import userdata
# API_KEY = userdata.get("GROQ_API_KEY")

# TODO: set API_KEY using ONE of the methods above.

# OpenAI-compatible client (works for Groq and OpenAI; Gemini users see their docs):
from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",   # remove this line if using OpenAI itself
)
MODEL = "llama-3.3-70b-versatile"                # or your provider's model name

print("Client ready.")

Client ready.


---
# Section 1 — Talking to an LLM Programmatically

Before building anything, understand the anatomy of an API call: **messages and roles**
(`system`, `user`, `assistant`), and the **generation parameters** (`temperature`,
`max_tokens`).

### Part 1.1 — Your first API call

In [22]:
# TODO: Write a helper function you will reuse for the WHOLE lab:
#
def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
    temperature=0.7, max_tokens=500):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response
#
# TODO: Call it once with a simple question and print the answer.
response = ask_llm("What is computer vision?")
print(response.choices[0].message.content)

# TODO: Print response.usage as well — how many tokens did your call consume?
print("\nToken usage:")
print(response.usage)

**Computer Vision: A Brief Overview**

Computer vision is a field of artificial intelligence (AI) that enables computers to interpret and understand visual information from the world. It combines computer science, mathematics, and engineering to develop algorithms and statistical models that allow computers to process, analyze, and understand digital images and videos.

**Key Aspects of Computer Vision:**

1. **Image Processing**: Enhancing, transforming, and manipulating images to extract relevant information.
2. **Object Detection**: Identifying and locating objects within images or videos.
3. **Image Classification**: Categorizing images into predefined classes or labels.
4. **Image Segmentation**: Dividing images into regions of interest or objects.
5. **Scene Understanding**: Interpreting the context and meaning of visual data.

**Applications of Computer Vision:**

1. **Self-Driving Cars**: Computer vision enables vehicles to detect and respond to their surroundings.
2. **Facial 

**Student Reasoning — Anatomy of a call**
*1. What is the difference between the `system` and `user` roles? Give an example of
something that belongs in each.*
*2. What is a token, roughly? Why do API providers bill per token rather than per request?*

> **Answer:** 1. The system role provides instructions that control how the LLM should behave, while the user role has the question or task being requested.                                                         2. A token is a piece of text an LLM can process for example a word or punctuation. API providers bill per token because different requests have very different input and generated text, which requires different number of computation.

### Part 1.2 — Temperature: the randomness dial

In [23]:
# TODO: Ask the SAME question 5 times at temperature=0.0 and 5 times at temperature=1.2.
#   A good test question: "Suggest a name for a savings product for market traders in Accra."
# TODO: Print all 10 answers, grouped by temperature.

question = "Suggest a name for a savings product for market traders in Accra."
#Temp = 0.0
print("Temperature 0.0")
for i in range(5):
    response = ask_llm(question, temperature=0.0)
    print(f"\nAnswer {i+1}:")
    print(response.choices[0].message.content)

print("Temperature 1.2")
for i in range(5):
    response = ask_llm(question, temperature=1.2)
    print(f"\nAnswer {i+1}:")
    print(response.choices[0].message.content)



Temperature 0.0

Answer 1:
Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **Trader's Treasure**: This name emphasizes the idea of saving and accumulating wealth.
3. **Sika Kokoo**: "Sika" means "money" in the Akan language, and "Kokoo" means "gather" or "collect". This name could appeal to market traders who want to gather and save their earnings.
4. **Market Mobi**: This name is short and catchy, and "Mobi" implies mobility and flexibility, which could be attractive to market traders who need to access their savings easily.
5. **Adanfo Save**: "Adanfo" means "friends" or "partners" in the Akan language, which could convey a sense of community and mutual support among market traders.
6. **Kae Dwa**: "Kae Dwa" means "keep and grow" in the Akan language, which could emphasize the idea of saving and growing one's wealth over time.
7. **Accra Tra

**Student Reasoning — Temperature**
*What did you observe at each temperature? For the loan decision-support system you are about
to build, which temperature regime is appropriate, and why?*

> **Answer:** At temperature 0.0, the responses were consistent as several suggestions were repeated across the five answers, with names such as “Makola Save” and “Trader's Treasure” appearing the most and at temperature 1.2, the responses varied, with more different names and ideas such as “Kelewele Savings,” “KioskKasa,” “TradeBoost,” and “Konko Savings.” Therefore, for the loan decision-support system, a lower temperature is more appropriate because financial decision support should be consistent and predictable instead of varying.

---
# Section 2 — The Dataset: Loan Application Letters

Run the next cell to load **six loan application letters** submitted to a (fictional)
microfinance institution in Ghana, plus **gold-standard extraction labels** for three of them
(you will use these for evaluation in Section 4).

Read at least two letters fully before moving on — you cannot engineer prompts for text you
have not read.

In [24]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.


---
# Section 3 — Prompt Engineering for the Decision Support System

You will now build the three components of the system, iterating on your prompts as you go.
**Keep every major prompt version** — Section 3.4 asks you to commit your prompt templates
and document how they evolved.

### Part 3.1 — Component 1: Summarization
Turn a rambling letter into a 3-4 sentence factual brief a busy loan officer can scan.

In [25]:
# TODO: Write SUMMARY_PROMPT_V1 — your first, naive attempt (e.g. just "Summarize this:").
#   Run it on L002 and L006. Read the output critically.

SUMMARY_PROMPT_V1 = "Summarize this:"

print("=== L002 - V1 === ")
response = ask_llm(f"{SUMMARY_PROMPT_V1}\n\n{LETTERS['L002']}")
print(response.choices[0].message.content)

print("\n=== L006 - V1 === ")
response = ask_llm(f"{SUMMARY_PROMPT_V1}\n\n{LETTERS['L006']}")
print(response.choices[0].message.content)

# TODO: Now write SUMMARY_PROMPT_V2 as a proper template with:
#   - a system prompt giving the LLM a ROLE (e.g. "You are an assistant to a microfinance
#     loan officer...") and constraints (factual, neutral, no invented details, 3-4 sentences)
#   - a user prompt template like: f"Summarize this loan application:\n\n{letter_text}"
#   Run V2 on the same two letters at temperature=0.

SUMMARY_SYSTEM_V2 = """You are an assistant to a microfinance loan officer.
Summarize loan applications factually and neutrally.
Use only information provided in the application and do not invent or assume details.
Keep the summary to 3-4 sentences."""

def summary_v2(letter_text):
    return ask_llm(
        f"Summarize this loan application: \n\n {letter_text}",
        system_prompt=SUMMARY_SYSTEM_V2,
        temperature=0
    )

print("\n=== L002 — V2 ===")
response = summary_v2(LETTERS["L002"])
print(response.choices[0].message.content)

print("\n=== L006 — V2 ===")
response = summary_v2(LETTERS["L006"])
print(response.choices[0].message.content)

# TODO: Compare V1 vs V2 outputs side by side. Keep both prompt versions in this notebook.

print("""
V1 showed reasonable summaries but was less controlled and sometimes introduced interpretations that were not explicitly stated in the letters, 
for example, it described Kofi as having “no experience,” although the letter only stated that he had not started the businesses yet. 
V2 was more factual and neutral because the system prompt specified the role, constraints, and sentence limit,
however, V2 still showed  a small unsupported interpretation for L002 by describing the applicant as seeking a “flexible repayment arrangement.” 
Overall, V2 is more suitable for a loan decision-support system because its outputs are more controlled, even though they still need to be checked for unsupported claims.
""")

=== L002 - V1 === 
Kwame Boateng, a commercial driver in Kumasi, is urgently seeking GHS 25,000 to repair his vehicle's engine and pay off debts. He's experiencing a slow business period but expects it to improve after the festive season. He has no collateral to offer but promises to repay the loan as soon as possible.

=== L006 - V1 === 
Kofi, a 22-year-old, is requesting GHS 50,000 to start three businesses: a car washing business, a provision shop, and a phone import business from Dubai. He has no experience or collateral, but claims to be "business-minded" and promises to repay the loan within a year when his businesses are successful, relying on his trustworthiness.

=== L002 — V2 ===
Kwame Boateng, a commercial driver in Kumasi, has applied for a loan of GHS 25,000. He intends to use the funds to repair his trotro engine and settle personal debts. Mr. Boateng mentions that his business has been slow, but he expects it to improve after the festive season. He does not have collater

**Student Reasoning — Summarization prompts**
*1. What concrete problems did V1's output have that V2 fixed? Quote examples.*
*2. Why is "no invented details" an essential instruction in this application? What is this
failure mode called in the LLM literature?*

> **Answer:** 1. V1 added interpretations that were not directly stated in the applications, for example, for L002, V1 said Kwame was “struggling due to slow business,” while the letter only says “Business has been slow.” For L006, V1 said Kofi had “no experience,” but the letter only says that he “has not started any of these yet.” Therefore V2 improved this by using factual and neutral instruction and to help avoid these stronger interpretations.                                                   2. "No invented details" is an essential instruction because this systems job is to support loan decisions and so unsupported information could make an applicant appear more or less suitable for a loan. This type of failure is called hallucination, where an LLM generates information that is not supported by the provided source.

### Part 3.2 — Component 2: Structured extraction (JSON)
Downstream software cannot read prose. Extract the fields in `GOLD` as strict JSON.

In [26]:
# TODO: Write EXTRACT_PROMPT — a template that instructs the model to return ONLY a JSON
#   object with EXACTLY these keys:
#     applicant_name (string), amount_ghs (number), purpose (string),
#     monthly_profit_ghs (number or null), has_collateral_or_guarantor (boolean),
#     repayment_months (number or null)
#   Techniques to use:
#     - explicit schema in the prompt
#     - ONE worked example (few-shot) using a letter you write yourself (not from LETTERS!)
#     - "If a field is not stated in the letter, use null. Do not guess."
#     - temperature=0

EXTRACT_PROMPT = """
You are an assistant helping a microfinance loan officer extract structured information
from loan application letters.

Return ONLY a valid JSON object with EXACTLY these keys:

{
  "applicant_name": "string",
  "amount_ghs": number,
  "purpose": "string",
  "monthly_profit_ghs": number or null,
  "has_collateral_or_guarantor": boolean,
  "repayment_months": number or null
}

Rules:
- Extract only information explicitly stated in the letter.
- If a field is not stated in the letter, use null.
- Do not guess or infer missing information.
- amount_ghs and monthly_profit_ghs must be numbers, not strings.
- has_collateral_or_guarantor must be true or false.
- repayment_months must be a number or null.
- Return ONLY the JSON object. Do not include explanations or markdown.

Worked example:

Letter:
"My name is Derek Hammond. I run a small bakery in Accra and have operated it
for three years. I am requesting GHS 10,000 to buy a new oven. I make GHS 3,200
profit each month. My sister will guarantee the loan. I will repay it over
10 months."

JSON:
{
  "applicant_name": "Derek Hammond",
  "amount_ghs": 10000,
  "purpose": "buy a new oven",
  "monthly_profit_ghs": 3200,
  "has_collateral_or_guarantor": true,
  "repayment_months": 10
}
"""

# TODO: Write extract_fields(letter_text) that calls the LLM, strips any ```json fences,
#   json.loads() the result, and returns a dict. Handle parse failures gracefully
#   (return None and print a warning).

response.choices[0].message.content

import json
import pandas as pd

def extract_fields(letter_text,temperature=0):
    try:
        response = ask_llm(
            f"{EXTRACT_PROMPT}\n\nLoan application letter:\n{letter_text}",
            temperature=temperature)

        result = response.choices[0].message.content.strip()

        if result.startswith("```json"):
            result = result[7:]
        elif result.startswith("```"):
            result = result[3:]
        if result.endswith("```"):
            result = result[:-3]

        result = result.strip()
        return json.loads(result)

    except (json.JSONDecodeError, Exception) as e:
        print(f"Warning: Could not parse extraction result: {e}")
        return None

# TODO: Run it on ALL SIX letters; collect results into a pandas DataFrame (one row per
#   letter) and display it.

results=[]

for letter_id, letter_text in LETTERS.items():
    extracted = extract_fields(letter_text)

    if extracted is not None:
        extracted["letter_id"] = letter_id
        results.append(extracted)

df = pd.DataFrame(results)
display(df)

,applicant_name,amount_ghs,purpose,monthly_profit_ghs,has_collateral_or_guarantor,repayment_months,letter_id
0,Akosua Mensah,8000,buy a deep freezer and expand into frozen foods,900.0,True,20.0,L001
1,Kwame Boateng,25000,repair my trotro engine and settle some person...,NaN,False,NaN,L002
2,Efua Darko,15000,purchase two industrial sewing machines and fa...,2800.0,True,15.0,L003
3,Yaw Owusu,12000,for feed and 500 new layers,1500.0,True,18.0,L004
4,Adenta Women's Weaving Cooperative,30000,buy a bulk order of yarn directly from the fac...,NaN,True,16.0,L005
5,Kofi,50000,"start a car washing business, a provision shop...",NaN,False,12.0,L006


**Student Reasoning — Structured extraction**
*1. Why must the few-shot example NOT come from the six letters you are processing?*
*2. Why "use null, do not guess" — what did the model do without that instruction?*
*3. Why is temperature=0 the right choice for extraction but arguably not for creative tasks?*

> **Answer:** 1.The few-shot example should not come from the original letters because those letters are the data we are testing the extraction prompt on and if we used one of them as the example, the model could learn or replicate information from the actual test data, making the evaluation less reliable. Therefore using a separate example shows that the prompt works on a new letter well rather than memorizing one of the answers.                                                                                                                                         2. “Use null, do not guess” is important because some information is not stated in the letters and without this instruction, the model may try to fill in missing information using assumptions. For example, L002 does not give a repayment period or monthly profit, so the correct extraction is null rather than an invented number and in our results, these missing fields are correctly returned as NaN in the DataFrame, which represent null.                                                                                                                                3. Temperature 0 is best for extraction because we want the model to produce consistent and predictable structured data from the same input, while creative tasks such as generating product names benefit from higher temperature because more variation and different ideas are useful. Therefore for extraction, creativity is undesirable because we want the model to reproduce the information in the letter accurately rather than generate different interpretations.

### Part 3.3 — Component 3: The decision-support brief
Combine everything: for each letter, produce a recommendation brief for the loan officer —
strengths, risks, missing information, and a suggested next step. The system must
**support** the decision, not **make** it.

In [27]:
# TODO: Write BRIEF_PROMPT — it receives the letter AND your extracted JSON, and must output:
#     1. Strengths (bullet points, grounded in the letter)
#     2. Risks / red flags (bullet points)
#     3. Missing information the officer should request
#     4. Suggested next step (e.g. "invite for interview", "request documents",
#        "flag for senior review") — NOT "approve" or "reject".
#   Give the model an explicit instruction that final decisions are made by humans.

BRIEF_PROMPT = """
You are an assistant to a microfinance loan officer.

Prepare a concise decision-support brief using ONLY the information provided
in the loan application letter and the extracted JSON.

Your output MUST contain exactly these four sections and nothing else:

1. Strengths
- Bullet points grounded in facts stated in the letter.
- Include only strengths that are directly supported by facts in the letter.
- Do not infer or assume strengths.
- Do not describe someone as experienced, profitable, financially capable,
  or having relevant skills unless the letter explicitly provides evidence.

2. Risks / red flags
- Bullet points based only on information in the letter.
- Include only risks or red flags that are directly supported by facts in
  the letter.
- Do not invent risks or make unsupported assumptions.
- Do not turn missing information into a risk; put it under Missing information.

3. Missing information the officer should request
- Use bullet points only.
- List information or documents that are not provided in the letter but
  would be useful for assessing the application.
- Do not invent facts to fill these gaps.

4. Suggested next step
- Give ONE appropriate process step, such as "invite for interview",
  "request documents", or "flag for senior review".
- Do NOT say "approve", "reject", "approve the loan", or "reject the loan".

Important:
- Final loan decisions are made by human loan officers, not by the LLM.
- The LLM provides decision support only.
- Do not make the final lending decision.
- Do not invent, infer, or assume facts.
- Do not repeat sections or provide "Step 1", "Step 2", etc.
- Do not add an introduction, conclusion, or explanation outside the
  four required sections.
- Be factual, neutral, concise, and grounded in the letter.

Loan application:
{letter_text}

Extracted information:
{extracted_json}
"""

# TODO: Generate briefs for ALL SIX letters. Print the briefs for L001, L002, and L006 —
#   three very different applications.

briefs = {}
for letter_id, letter_text in LETTERS.items():
    extracted = next(
        row for row in results
        if row["letter_id"] == letter_id
    )

    extracted_json = json.dumps(extracted, indent=2)
    response = ask_llm(
        BRIEF_PROMPT.format(
            letter_text=letter_text,
            extracted_json=extracted_json),temperature=0)

    briefs[letter_id] = response.choices[0].message.content

for letter_id in ["L001", "L003", "L006"]:
    print(f"\n{'='*60}")
    print(f"BRIEF — {letter_id}")
    print(f"{'='*60}")
    print(briefs[letter_id])


BRIEF — L001
1. Strengths
- The applicant has been selling provisions at Makola Market for 12 years.
- The applicant's current stall makes about GHS 900 profit each month.
- The applicant has saved GHS 2,500 with the susu scheme over the past two years and has never missed a contribution.
- The applicant has a guarantor, her sister, who is a teacher.

2. Risks / red flags
- The applicant is requesting a loan that is significantly larger than her current monthly profit.

3. Missing information the officer should request
- Detailed business plan for the expansion into frozen foods
- Proof of the applicant's sister's employment as a teacher
- Information about the applicant's current expenses and debt obligations
- Valuation of the deep freezer to be purchased

4. Suggested next step
- Request documents

BRIEF — L003
1. Strengths
- The business is registered (registration no. BN-2019-4482).
- The applicant has a fixed deposit of GHS 5,000 with GCB that can be pledged as collateral.
- The

**Student Reasoning — Decision support**
*1. Compare the briefs for L003 (strong application) and L006 (weak application). Did the
system identify the right strengths and red flags in each?*
*2. Why did we forbid the model from outputting "approve"/"reject"? Give one practical and
one ethical reason.*

> **Answer:** 

### Part 3.4 — Commit your prompt templates
Prompts ARE code. Save your final `SUMMARY_PROMPT`, `EXTRACT_PROMPT`, and `BRIEF_PROMPT` into
a separate file `prompts.py` (or `prompts.md`) in your repository and commit it with a
message describing how the prompts evolved. Paste your commit hash below.

> **Commit hash:** 6166acb

---
# Section 4 — Evaluation: Quality, Reliability, Appropriateness

An impressive demo is not a trustworthy system. Now measure it.

### Part 4.1 — Extraction accuracy against gold labels

In [28]:
# TODO: For the three letters in GOLD, compare your extracted DataFrame to the gold values
#   field by field. Compute per-field accuracy across the three letters
#   (name matching can be case-insensitive; numbers must match exactly).

fields = [
    "applicant_name",
    "amount_ghs",
    "purpose",
    "monthly_profit_ghs",
    "has_collateral_or_guarantor",
    "repayment_months"
]

gold_ids = ["L001", "L003", "L006"]
comparison = []

for field in fields:
    row = {"field": field}
    correct_count = 0

    for letter_id in gold_ids:
        extracted_value = df.loc[
            df["letter_id"] == letter_id, field].iloc[0]
        
        gold_value = GOLD[letter_id][field]

        if field == "applicant_name":
            correct = str(extracted_value).strip().lower() == str(gold_value).strip().lower()

        elif gold_value is None:
            correct = pd.isna(extracted_value)

        else:
            correct = extracted_value == gold_value

        row[letter_id] = "✓" if correct else "✗"
        if correct:
            correct_count += 1

    row["accuracy"] = correct_count / len(gold_ids)

    comparison.append(row)

# TODO: Display a small table: rows = fields, columns = L001 / L003 / L006 / accuracy.
comparison_df = pd.DataFrame(comparison)

display(comparison_df)

,field,L001,L003,L006,accuracy
0,applicant_name,✓,✓,✓,1.0
1,amount_ghs,✓,✓,✓,1.0
2,purpose,✗,✗,✗,0.0
3,monthly_profit_ghs,✓,✓,✓,1.0
4,has_collateral_or_guarantor,✓,✓,✓,1.0
5,repayment_months,✓,✓,✓,1.0


### Part 4.2 — Reliability: is the system consistent?

In [29]:
# TODO: Run extract_fields() on letter L004 FIVE times at temperature=0 and FIVE times at
#   temperature=1.0.

letter = LETTERS["L004"]
results_temp0 = []
results_temp1 = []

for i in range(5):
    result = extract_fields(letter, temperature=0)
    results_temp0.append(result)

for i in range(5):
    result = extract_fields(letter, temperature=1.0)
    results_temp1.append(result)

# TODO: For each temperature, report how many of the 5 runs produced (a) valid JSON and
#   (b) identical values across runs. A simple approach: json.dumps(result, sort_keys=True)
#   and count unique strings.

def evaluate_runs(results):
    valid_results = [r for r in results if r is not None]

    unique_results = set(
        json.dumps(r, sort_keys=True)
        for r in valid_results)

    return len(valid_results), len(unique_results)

valid0, unique0 = evaluate_runs(results_temp0)
valid1, unique1 = evaluate_runs(results_temp1)
print("Temperature = 0.0")
print(f"Valid JSON: {valid0}/5")
print(f"Unique outputs: {unique0}")

print("\nTemperature = 1.0")
print(f"Valid JSON: {valid1}/5")
print(f"Unique outputs: {unique1}")

Temperature = 0.0
Valid JSON: 5/5
Unique outputs: 1

Temperature = 1.0
Valid JSON: 5/5
Unique outputs: 1


### Part 4.3 — Hallucination probing

In [ ]:
# TODO: Design TWO adversarial tests and run them:
#   Test 1 — Ask your summarizer a question about a detail that is NOT in a letter
#     (e.g. "What is the applicant's credit score?"). Does it admit the information is
#     absent, or does it invent one?
#   Test 2 — Feed your extractor an EMPTY or IRRELEVANT text (e.g. a weather report).
#     Does it return nulls, or does it fabricate an applicant?

test1_prompt = f"""
Summarize this loan application.

Answer this question:
What is the applicant's credit score?

If the credit score is not stated in the letter, explicitly say that it is not provided.
Do not guess or invent a credit score.

Loan application:
{LETTERS["L002"]}
"""

response_test1 = ask_llm(
    test1_prompt,
    system_prompt=SUMMARY_SYSTEM_V2,
    temperature=0)

test1_output = response_test1.choices[0].message.content

weather_report = """
The weather in Accra today will be partly cloudy with temperatures
between 24 and 30 degrees Celsius. There may be occasional rainfall
in the afternoon, with moderate winds expected.
"""
test2_output = extract_fields(weather_report)

# TODO: Record the outputs verbatim below and label each PASS or FAIL.
print("=== TEST 1 RESULT ===")
print(test1_output)

print("\n===== TEST 2 RESULT =====")
print(json.dumps(test2_output, indent=2))

print("""
Test 1: PASS
Reason: The model correctly stated that the credit score was not provided instead of inventing a value.

Test 2: PASS
Reason: The model returned null for all fields because the weather report contained no loan application information. 
""")

=== TEST 1 RESULT ===
Kwame Boateng, a commercial driver in Kumasi, has applied for a loan of GHS 25,000 to repair his trotro engine and settle personal debts. He expects his business to improve after the festive season and is willing to repay the loan when he can. The applicant does not have collateral to offer at this time. The credit score of the applicant is not provided in the loan application.

===== TEST 2 RESULT =====
{
  "applicant_name": null,
  "amount_ghs": null,
  "purpose": null,
  "monthly_profit_ghs": null,
  "has_collateral_or_guarantor": null,
  "repayment_months": null
}

Test 1: PASS
Reason: The model correctly stated that the credit score was not provided instead of inventing a value.

Test 2: PASS
Reason: The model returned null for all fields because the weather report contained no loan application information. 



**Student Reasoning — Evaluation results**
*1. Report your extraction accuracy. Which field was hardest for the model and why?*
*2. What did the reliability experiment show about temperature and production systems?*
*3. Did your system hallucinate under probing? If yes, how could the prompt (or the system
design around it) reduce the risk?*

> **Answer:** All the fields had an accuracy of 1.0 except the Purpose field which had an accuracy of 0. The purpose field was the hardest because it had to make sure that the prompt we engineered and what python engineered match word for word and that won't always necessarily happen and that is why the accuracy for purpose was 0.                                                                                                                                               2. The reliability experiment showed that even with a change in temperature the JSON and outputs remained the same showing that even if you allow more creativity 

### Part 4.4 — Appropriateness: should this system exist?
No code in this part — just judgment, which is the scarcest skill in AI for business.

**Student Reasoning — Appropriateness**
*1. Letters L002 and L006 would likely be declined. If the bank fully automated decisions
with your system, who could be unfairly harmed, and how? Consider applicants who write
poorly in English but run solid businesses.*
*2. Loan letters contain personal data. What are the implications of sending them to a
third-party API in another country? What would you check before deploying this at a real
Ghanaian microfinance institution?*
*3. Name TWO concrete safeguards you would build around this system in production (think:
human review points, logging, appeal processes, monitoring).*

> **Answer:** [Double-click to edit]

---
# Section 5 — Reflection

*Answer in a few sentences each:*

1. **Prompting as engineering:** How is iterating on a prompt similar to and different from
   iterating on the model hyperparameters you tuned in Lab 3?
2. **Trust:** After your Section 4 evaluation, would you trust this system to run unattended?
   What single evaluation result most influenced your answer?
3. **Cost and scale:** Estimate (from your `response.usage` numbers) the tokens needed to
   process 1,000 applications per month. What does that imply for provider choice?
4. **Looking back at the course:** You have now used classical ML (Lab 2), trained neural
   networks (Lab 3), and used a foundation model via API (Lab 4). For a task like this one,
   why does calling an API beat training your own model — and when would it not?

> **Answer:** [Double-click to edit]

---
### Submission checklist

- [ ] All cells run top-to-bottom with no errors (`Kernel -> Restart & Run All`).
- [ ] **No API key anywhere in the notebook or the commit history.**
- [ ] Every **Student Reasoning** box is filled in with full sentences.
- [ ] `prompts.py` / `prompts.md` committed with your final prompt templates.
- [ ] Evaluation tables and adversarial test outputs visible in the saved notebook.
- [ ] Notebook pushed to `lab-4-llm-decision-support` with incremental commits.
- [ ] Repository link submitted to the course portal.
- [ ] AI Declaration form in Repository.